In [ ]:
# All imports used by this notebook.
from pathlib import Path
import importlib.util
import inspect
import os
import shutil
import subprocess
import sys
from pprint import pprint
from typing import Any

import yaml


def import_module_from_path(module_name: str, path: Path):
    """Import a Python module from an explicit file path, always reloading from disk."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Module file not found: {path}")

    # Avoid accidentally reusing an older imported converter from sys.modules.
    sys.modules.pop(module_name, None)

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for {path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def add_to_sys_path(path: Path) -> None:
    """Prepend a path to sys.path if it is not already present."""
    path_str = str(Path(path).resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)


def ensure_fsp_by_resp_id(path: Path) -> bool:
    """
    Ensure a ChainScope CotResponses YAML file contains fsp_by_resp_id.

    For these Qwen generations there were no few-shot prompts, so the value is null.
    Returns True if the file was patched.
    """
    path = Path(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML mapping in {path}")

    if "fsp_by_resp_id" in data:
        return False

    data["fsp_by_resp_id"] = None
    path.write_text(
        yaml.safe_dump(data, sort_keys=False, allow_unicode=True, width=120),
        encoding="utf-8",
    )
    return True


# Convert Qwen JSONL responses to ChainScope YAML, then run `eval_cots.py`

This notebook is organized as:

1. Configure conversion
2. Run conversion
3. Configure eval
4. Run eval

It expects this local layout by default:

```text
REPO_ROOT/
  chainscope/
    scripts/iphr/eval_cots.py
  convert_qwen_jsonl_to_chainscope_yaml.py
  selected_621_pairs.json
  data1/modal_qwen3_4b_backup_20260710_0015/outputs/chunks/chunk_0000.jsonl
```

For no-few-shot runs, ChainScope still requires this top-level field in every `CotResponses` YAML:

```yaml
fsp_by_resp_id: null
```

The converter writes it, and this notebook also patches it before validation/eval if an older YAML is encountered.


## 1. Configure conversion

In [ ]:
# Set this to the directory that contains:
#   - chainscope/
#   - convert_qwen_jsonl_to_chainscope_yaml.py
#   - selected_621_pairs.json
#   - data1/
#
# If this notebook is already running from that directory, Path.cwd() is correct.
REPO_ROOT = Path.cwd().resolve()

# ChainScope checkout inside REPO_ROOT.
CHAIN_SCOPE_ROOT = REPO_ROOT / "chainscope"

# Make your local project and ChainScope importable.
add_to_sys_path(REPO_ROOT)
add_to_sys_path(CHAIN_SCOPE_ROOT)

# Converter script generated by ChatGPT. Put it in REPO_ROOT, or edit this path.
CONVERTER_PY = REPO_ROOT / "convert_qwen_jsonl_to_chainscope_yaml.py"

# Input files.
RESPONSES_JSONL = REPO_ROOT / "data1/modal_qwen3_4b_backup_20260710_0015" / "outputs" / "chunks" / "chunk_0000.jsonl"
GOLD_JSON = REPO_ROOT / "selected_621_pairs.json"
EVAL_COTS_PY = REPO_ROOT / "chainscope" / "scripts" / "iphr" / "eval_cots.py"

# Conversion output.
# This can be anywhere writable. It does not have to be inside chainscope/data.
CONVERSION_OUT_DIR = REPO_ROOT / "converted_qwen_chainscope"

# Converter options.
INSTR_ID = "instr-wm"
CONVERSION_SUFFIX = "converted"
REPO_LAYOUT = False
WRITE_QUESTION_YAMLS = True

# Recommended for reruns after changing converter/schema.
# This deletes the conversion output directory before writing fresh YAMLs.
CLEAR_OUTPUT_DIR = True

print("REPO_ROOT:", REPO_ROOT)
print("CHAIN_SCOPE_ROOT:", CHAIN_SCOPE_ROOT)
print("CONVERTER_PY:", CONVERTER_PY)
print("RESPONSES_JSONL:", RESPONSES_JSONL)
print("GOLD_JSON:", GOLD_JSON)
print("EVAL_COTS_PY:", EVAL_COTS_PY)
print("CONVERSION_OUT_DIR:", CONVERSION_OUT_DIR)


## 2. Run conversion

In [ ]:
# Basic path checks before conversion.
required_paths = {
    "REPO_ROOT": REPO_ROOT,
    "CHAIN_SCOPE_ROOT": CHAIN_SCOPE_ROOT,
    "CONVERTER_PY": CONVERTER_PY,
    "RESPONSES_JSONL": RESPONSES_JSONL,
    "GOLD_JSON": GOLD_JSON,
}

missing = [f"{name}: {path}" for name, path in required_paths.items() if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))

if CLEAR_OUTPUT_DIR and CONVERSION_OUT_DIR.exists():
    shutil.rmtree(CONVERSION_OUT_DIR)
    print(f"Deleted old conversion output: {CONVERSION_OUT_DIR}")

converter_module = import_module_from_path("convert_qwen_jsonl_to_chainscope_yaml", CONVERTER_PY)

if hasattr(converter_module, "convert_qwen_jsonl_to_chainscope_yaml"):
    summary = converter_module.convert_qwen_jsonl_to_chainscope_yaml(
        responses_path=RESPONSES_JSONL,
        gold_path=GOLD_JSON,
        out_dir=CONVERSION_OUT_DIR,
        instr_id=INSTR_ID,
        suffix=CONVERSION_SUFFIX,
        repo_layout=REPO_LAYOUT,
        write_question_yamls=WRITE_QUESTION_YAMLS,
    )
else:
    # Fallback for an older CLI-only converter.
    cmd = [
        sys.executable,
        str(CONVERTER_PY),
        "--responses", str(RESPONSES_JSONL),
        "--gold", str(GOLD_JSON),
        "--out-dir", str(CONVERSION_OUT_DIR),
        "--instr-id", INSTR_ID,
        "--suffix", CONVERSION_SUFFIX,
    ]
    if REPO_LAYOUT:
        cmd.append("--repo-layout")
    if WRITE_QUESTION_YAMLS:
        cmd.append("--write-question-yamls")
    print("Converter module has no function; using CLI fallback:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    manifest_path = CONVERSION_OUT_DIR / "response_paths.txt"
    summary = {
        "rows_read": None,
        "gold_pairs_loaded": None,
        "rows_skipped_unknown_pair_id": None,
        "cot_response_paths": [
            line.strip()
            for line in manifest_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ],
        "question_paths": [],
        "manifest_path": str(manifest_path),
    }

response_paths = [Path(p) for p in summary["cot_response_paths"]]
manifest_path = Path(summary["manifest_path"])

# Extra safety: patch already-written YAML files if an older converter omitted fsp_by_resp_id.
patched = [p for p in response_paths if ensure_fsp_by_resp_id(p)]

print(f"Read response rows: {summary['rows_read']}")
print(f"Loaded golden pairs: {summary['gold_pairs_loaded']}")
print(f"Skipped rows with unknown pair_id: {summary['rows_skipped_unknown_pair_id']}")
print(f"Wrote CotResponses YAML files: {len(response_paths)}")
print(f"Patched YAML files missing fsp_by_resp_id: {len(patched)}")
print(f"Wrote question YAML files: {len(summary.get('question_paths', []))}")
print(f"Manifest: {manifest_path}")

print("\nFirst few response paths:")
for p in response_paths[:5]:
    print(" ", p)

# Confirm every generated response YAML contains the required field.
missing_required = []
for p in response_paths:
    data = yaml.safe_load(Path(p).read_text(encoding="utf-8"))
    if not isinstance(data, dict) or "fsp_by_resp_id" not in data:
        missing_required.append(p)

if missing_required:
    raise RuntimeError("Some files still lack fsp_by_resp_id:\n" + "\n".join(map(str, missing_required)))

print("Confirmed fsp_by_resp_id exists in every generated CotResponses YAML.")


In [ ]:
# Validate generated YAMLs against your local ChainScope CotResponses dataclass.
# This catches schema mismatches before running eval_cots.py.
from chainscope.typing import CotResponses

loaded_responses = []
for p in response_paths:
    # Defensive patch in case this cell is rerun against older outputs.
    ensure_fsp_by_resp_id(p)
    loaded_responses.append(CotResponses.load(p))

print(f"Validated {len(loaded_responses)} CotResponses YAML files")

summary_rows = []
for cr, path in zip(loaded_responses, response_paths):
    n_questions = len(cr.responses_by_qid)
    n_responses = sum(len(v) for v in cr.responses_by_qid.values())
    summary_rows.append({
        "path": str(path),
        "model_id": cr.model_id,
        "prop_id": cr.ds_params.prop_id,
        "comparison": cr.ds_params.comparison,
        "answer": cr.ds_params.answer,
        "n_questions": n_questions,
        "n_responses": n_responses,
        "fsp_by_resp_id": getattr(cr, "fsp_by_resp_id", None),
    })

pprint(summary_rows[:10])


## 3. Configure eval

In [ ]:
# Use the response paths produced by the conversion step.
EVAL_RESPONSE_PATHS = response_paths

# Heuristic-only evaluation:
EVALUATOR_MODEL_ID = None
EVAL_API = None

# Paper-style LLM judge example:
# EVALUATOR_MODEL_ID = "C3.7S"
# EVAL_API = "ant"
# Make sure ANTHROPIC_API_KEY is set in your environment for --api ant.

EVAL_TEST_MODE = False
EVAL_VERBOSE = True

print(f"Configured {len(EVAL_RESPONSE_PATHS)} response YAML files for eval")
print("EVALUATOR_MODEL_ID:", EVALUATOR_MODEL_ID)
print("EVAL_API:", EVAL_API)
print("EVAL_TEST_MODE:", EVAL_TEST_MODE)
print("EVAL_VERBOSE:", EVAL_VERBOSE)

if EVAL_API == "ant":
    print("ANTHROPIC_API_KEY present:", bool(os.environ.get("ANTHROPIC_API_KEY")))
if EVAL_API == "oai":
    print("OPENAI_API_KEY present:", bool(os.environ.get("OPENAI_API_KEY")))


## 4. Run `eval_cots.py`

In [ ]:
if not EVAL_COTS_PY.exists():
    raise FileNotFoundError(f"eval_cots.py not found: {EVAL_COTS_PY}")

# Defensive patch before eval in case files were edited between cells.
patched_before_eval = [p for p in EVAL_RESPONSE_PATHS if ensure_fsp_by_resp_id(p)]
if patched_before_eval:
    print(f"Patched {len(patched_before_eval)} files before eval.")

eval_cots_module = import_module_from_path("chainscope_eval_cots_script", EVAL_COTS_PY)

# eval_cots.py uses click, so submit is usually a click Command.
# The actual Python function is available as submit.callback.
callback = getattr(eval_cots_module.submit, "callback", eval_cots_module.submit)
signature = inspect.signature(callback)

responses_arg = ",".join(str(p) for p in EVAL_RESPONSE_PATHS)

candidate_kwargs = {
    "responses_paths": responses_arg,
    "verbose": EVAL_VERBOSE,
    "llm_model_id": EVALUATOR_MODEL_ID,
    "api": EVAL_API,
    "test": EVAL_TEST_MODE,
}

call_kwargs = {
    name: value
    for name, value in candidate_kwargs.items()
    if name in signature.parameters
}

print("eval_cots callback signature:", signature)
print("Calling eval_cots with kwargs:")
pprint(call_kwargs)

callback(**call_kwargs)

print("Evaluation finished.")
